# Phase 6: API + Dashboard + Deployment
**Environment:** Google Colab (T4 GPU) + cloudflared Tunnel

**Framework:** FastAPI + Uvicorn

**Goal:** Serve Mask R-CNN inference via REST API accessible from anywhere

## 1. Install Dependencies
Install FastAPI, Uvicorn, ngrok, and Detectron2 on Colab environment.

Restart runtime after this cell completes.

In [1]:
!pip install fastapi uvicorn python-multipart pyngrok nest-asyncio
!pip install pyyaml==5.1
!pip install 'git+https://github.com/facebookresearch/detectron2.git'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.2/274.2 kB 11.9 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Cloning https://github.com/facebookresearch/detectron2.git to /tmp/pip-req-build-8xq_upll
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/detectron2.git /tmp/pip-req-build-8xq_upll
  Resolved https://github.com/facebookresearch/detectron2.git to commit 8a9d885b3d4dcf1bef015f0593b872ed8d32b4ab
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.

## 2. Mount Google Drive
Mount Drive to access model weights and inference script stored from Phase 2.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Verify model exists
MODEL_PATH = '/content/drive/MyDrive/waste-intelligence/models/final/waste_mask_rcnn_final.pth'
print('Model exists:', os.path.exists(MODEL_PATH))
print('Drive contents:')
print(os.listdir('/content/drive/MyDrive/waste-intelligence'))

Mounted at /content/drive
Model exists: True
Drive contents:
['data', 'models', 'src', '02_modeling.ipynb', '06_api.ipynb']


## 3. Install and Setup cloudflared Tunnel
Install cloudflared to expose local FastAPI server to public internet.

No account required — free tunnel via Cloudflare Tunnel.

Replaces ngrok which had installation issues on Colab.

In [3]:
# Install cloudflared instead of ngrok
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print('cloudflared installed successfully')

cloudflared installed successfully


## 4. Import Inference Module
Add project path to sys.path and import inference functions from Phase 2.

Verify WASTE_CLASSES are loaded correctly.

In [4]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/waste-intelligence')

from src.vision.inference import load_predictor, run_inference, WASTE_CLASSES
print('Inference module loaded successfully')
print('Classes:', WASTE_CLASSES)

Inference module loaded successfully
Classes: ['Metal', 'Mixed Waste', 'Paper-Cardboard', 'Plastic', 'Wood']


## 5. Define FastAPI Application
Define FastAPI app with CORS middleware and implement endpoints.

POST /analyze accepts image upload and returns waste composition per class.

Model is loaded once at startup and reused for all requests.

In [5]:
import nest_asyncio
import uvicorn
import tempfile
import os
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from datetime import datetime

app = FastAPI(
    title='Industrial Waste Intelligence API',
    description='End-to-end AI platform for industrial waste composition analysis',
    version='1.0.0'
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*']
)

# Load model once at startup
print('Loading Mask R-CNN model...')
predictor = load_predictor(MODEL_PATH, score_thresh=0.3)
print('Model loaded successfully')

@app.get('/health')
async def health():
    """Lightweight health check."""
    return {
        'status'   : 'ok',
        'model'    : 'Mask R-CNN ResNet-50 FPN',
        'mAP'      : '48.95%',
        'timestamp': datetime.now().isoformat()
    }

@app.post('/analyze')
async def analyze(file: UploadFile = File(...)):
    """
    Upload a waste image and return segmentation results
    with percentage composition per waste class.
    """
    # Validate file type
    if not file.filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        raise HTTPException(status_code=400, detail='Only JPG and PNG images are supported')

    # Save uploaded file to temp location
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as tmp:
        tmp.write(await file.read())
        tmp_path = tmp.name

    # Run inference
    instances, composition, _ = run_inference(predictor, tmp_path)
    os.unlink(tmp_path)

    return {
        'filename'           : file.filename,
        'instances'          : len(instances),
        'composition'        : composition,
        'total_area_detected': round(sum(composition.values()), 2)
    }

print('FastAPI app defined successfully')
print('Endpoints:')
print('  GET  /health')
print('  POST /analyze')

Loading Mask R-CNN model...
Model loaded successfully
FastAPI app defined successfully
Endpoints:
  GET  /health
  POST /analyze


## 6. Start API Server + cloudflared Tunnel
Start Uvicorn server on port 8000 and expose via cloudflare public URL.

Copy the trycloudflare.com URL for use in Streamlit dashboard.

Note: URL changes every Colab session restart — update Streamlit config accordingly.

In [ ]:
import subprocess
import threading
import time
import re
import nest_asyncio
import uvicorn
import asyncio

nest_asyncio.apply()

# Start cloudflared tunnel
def start_tunnel():
    process = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
        stderr=subprocess.PIPE,
        stdout=subprocess.PIPE
    )
    for line in process.stderr:
        line = line.decode()
        if 'trycloudflare.com' in line:
            match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
            if match:
                print(f'Public URL : {match.group(0)}')
                print(f'API docs   : {match.group(0)}/docs')
                print(f'Health     : {match.group(0)}/health')
                print('\nCopy the Public URL above for Streamlit dashboard')
                break

tunnel_thread = threading.Thread(target=start_tunnel)
tunnel_thread.daemon = True
tunnel_thread.start()

time.sleep(3)

# Start server
config = uvicorn.Config(app, host='0.0.0.0', port=8000)
server = uvicorn.Server(config)
await server.serve()

INFO:     Started server process [1744]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL : https://sisters-demonstrates-advert-alternatives.trycloudflare.com
API docs   : https://sisters-demonstrates-advert-alternatives.trycloudflare.com/docs
Health     : https://sisters-demonstrates-advert-alternatives.trycloudflare.com/health

Copy the Public URL above for Streamlit dashboard
INFO:     118.6.113.115:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     118.6.113.115:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     118.6.113.115:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     118.6.113.115:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     118.6.113.115:0 - "GET /health HTTP/1.1" 200 OK


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0331 01:29:53.474000 1744 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


INFO:     118.6.113.115:0 - "POST /analyze HTTP/1.1" 200 OK
